# AnyMatch on Colab Pro (A100)

End-to-end notebook to train AnyMatch on the 9 public EM datasets and run inference on AllianceChicago patient pairs.

**Before running**: Runtime → Change runtime type → **A100 GPU**.

**PHI reminder**: do not paste raw patient rows into Colab Free; use Colab Pro / Workspace per your institution's policy.

## 1. Mount Drive and bring in the code

Easiest workflow: zip the local `AnyMatch/` folder (with the patches already applied) and drop it in `MyDrive/AnyMatch/`. The cell below assumes you've placed `AnyMatch.zip` in your Drive root.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, shutil
WORK = '/content/anymatch'
# Path to the zip on Drive (edit if you put it in a subfolder).
DRIVE_ZIP = '/content/drive/MyDrive/AnyMatch.zip'
DRIVE_CKPT_DIR = '/content/drive/MyDrive/AnyMatch/checkpoints'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

!rm -rf {WORK} && mkdir -p {WORK}
!unzip -q {DRIVE_ZIP} -d {WORK}

# If the zip nested everything under AnyMatch/, flatten one level and remove the empty dir.
inner = glob.glob(f'{WORK}/*/loo.py')
if inner:
    src = os.path.dirname(inner[0])
    !cp -a {src}/. {WORK}/
    shutil.rmtree(src, ignore_errors=True)

%cd {WORK}
!ls

Mounted at /content/drive
/content/anymatch
anymatch_colab.ipynb  diagram	       model.py		    string_simlarity.py
anymatch.png	      environment.yml  predict_alliance.py  utils
data		      inference.py     README.md
data.py		      loo.py	       SETUP.md


## 2. Install dependencies

Colab ships with PyTorch + CUDA. We just need `transformers`, `autogluon` (used by `automl_filter`), and a few utilities. Pin `transformers` to avoid breaking changes.

In [2]:
# !pip install -q 'transformers==4.36.2' 'tokenizers<0.20' 'datasets<3' 'scikit-learn' 'pandas' 'tqdm'
# autogluon is heavy; only needed if you re-run the AutoML pre-filtering inside preprocess.ipynb.
# If your data/prepared/<dataset>/ folders already contain the AutoML output CSVs, skip this.
# !pip install -q autogluon.tabular

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 161.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


## 3. Verify prepared data is in place

`AnyMatch.zip` already contains `data/prepared/` for all 9 datasets, so this cell normally just confirms it. Fallback order:

1. Use `data/prepared/` from the unzipped folder (default).
2. If missing, sync from `MyDrive/AnyMatch/prepared/`.
3. If neither, run `data/preprocess.ipynb` to build it.

In [2]:
LOCAL_PREPARED = 'data/prepared'
DRIVE_PREPARED = '/content/drive/MyDrive/AnyMatch/prepared'

if os.path.isdir(LOCAL_PREPARED) and any(os.scandir(LOCAL_PREPARED)):
    print(f'Using prepared data already unzipped at {LOCAL_PREPARED}/')
    !ls {LOCAL_PREPARED}
elif os.path.isdir(DRIVE_PREPARED):
    print(f'Syncing prepared data from {DRIVE_PREPARED} ...')
    !mkdir -p {LOCAL_PREPARED}
    !rsync -a {DRIVE_PREPARED}/ {LOCAL_PREPARED}/
    !ls {LOCAL_PREPARED}
else:
    print('No prepared data found locally or on Drive.')
    print('Run data/preprocess.ipynb (locally or here) and either include data/prepared/')
    print(f'in AnyMatch.zip, or upload it to {DRIVE_PREPARED}.')

Using prepared data already unzipped at data/prepared/
abt  amgo  beer  dbac  dbgo  foza  itam  waam  wdc


## 4. Train AnyMatch on all 9 datasets

This uses the patched `loo.py`: `--leaved_dataset_name none` trains on all 9, and `--save_model_path` persists the best checkpoint.

On an A100 with GPT-2 + paper defaults, expect roughly 30–60 minutes.

In [4]:
!pip install autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 29.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is stil

In [5]:
CKPT_DIR = 'saved_models/anymatch_all9_gpt2'
# Default uses --row_sample_func one_pos_two_neg to skip the autogluon dependency.
# Swap to --row_sample_func automl_filter for the paper's best config (requires
# uncommenting the AutoML cell in data/preprocess.ipynb and installing autogluon).
!python loo.py \
    --seed 42 \
    --base_model gpt2 \
    --leaved_dataset_name none \
    --serialization_mode mode1 \
    --train_data attr+row \
    --row_sample_func one_pos_two_neg \
    --patience_start 20 \
    --save_model_path {CKPT_DIR}

2026-05-20 22:25:06.386541: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-20 22:25:06.457570: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
config.json: 100% 665/665 [00:00<00:00, 4.57MB/s]
model.safetensors: 100% 548M/548M [00:02<00:00, 190MB/s] 
Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
tokenizer_

In [6]:
# Back up the checkpoint to Drive so it survives runtime disconnect.
!rsync -a {CKPT_DIR}/ {DRIVE_CKPT_DIR}/anymatch_all9_gpt2/
!ls {DRIVE_CKPT_DIR}/anymatch_all9_gpt2/

config.json  model.safetensors	      tokenizer_config.json
merges.txt   special_tokens_map.json  vocab.json


## 5. Inference on AllianceChicago patient pairs

Upload your patient pairs CSV (columns suffixed `_l` and `_r`, plus an optional `label`) to Drive at `MyDrive/AnyMatch/alliance/pairs.csv`.

The script writes back a CSV with `pred` and `match_prob` columns appended.

In [ ]:
ALLIANCE_IN = '/content/drive/MyDrive/AnyMatch/alliance/pairs.csv'
ALLIANCE_OUT = '/content/drive/MyDrive/AnyMatch/alliance/predictions.csv'
!python predict_alliance.py \
    --model_path {CKPT_DIR} \
    --base_model gpt2 \
    --serialization_mode mode1 \
    --input_csv {ALLIANCE_IN} \
    --output_csv {ALLIANCE_OUT} \
    --batch_size 64

## 6. Sanity-check on a public test set

Before trusting the patient predictions, run the trained model on a Magellan test split (e.g. `abt_buy`) and confirm F1 is in the paper's reported range. This validates the training + inference pipeline end-to-end.

In [7]:
import pandas as pd
from transformers import GPT2Tokenizer, GPT2ForSequenceClassification
from data import GPTDataset
from utils.data_utils import read_single_row_data
from utils.train_eval import inference

tokenizer = GPT2Tokenizer.from_pretrained(CKPT_DIR)
model = GPT2ForSequenceClassification.from_pretrained(CKPT_DIR)
model.config.pad_token_id = model.config.eos_token_id

_, _, test_df = read_single_row_data('data/prepared/abt', mode='mode1', print_info=False)
test_d = GPTDataset(tokenizer, test_df, max_len=10000)
f1, acc = inference(tokenizer, model, test_d, batch_size=128, base_model='gpt2')
print(f'abt_buy F1={f1*100:.2f} acc={acc*100:.2f}')

0 out of 1916 data samples are filtered.
The predictions and ground truth are:
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,